In [2]:

import os
import shutil
from pathlib import Path
import pandas as pd
import re

# === Parâmetros principais ===
ORDER = 2  # 1 para 1ª ordem, 2 para 2ª ordem
IMAGES_DIR = Path("./artifacts")  # onde estão as imagens originais
EXT = ".png"  # extensão dos arquivos

# === Caminhos automáticos ===
LABELS_CSV = IMAGES_DIR / f"user_cluster_labels_umap_order{ORDER}.csv"
OUT_DIR = Path(f"./clusters_{ORDER}order")

print(f"Usando arquivo de labels: {LABELS_CSV}")
print(f"Pasta de saída: {OUT_DIR}")

# === Função auxiliar ===
def build_image_name(user_id: str, order: int, ext: str = ".png") -> str:
    return f"user_{user_id}_order{order}_fixed{ext}"

# === Leitura do CSV ===
df = pd.read_csv(LABELS_CSV)
print(f"Registros carregados: {len(df):,}")

# === Criar pastas de saída ===
OUT_DIR.mkdir(parents=True, exist_ok=True)
clusters = sorted(df["cluster"].unique().tolist())
for c in clusters:
    (OUT_DIR / f"cluster_{c}").mkdir(parents=True, exist_ok=True)
    cluster_src = IMAGES_DIR / f"cluster_{c}_mean_heatmap_umap_order{ORDER}.png"  # cluster_0_mean_heatmap_umap_order1
    cluster_dst = OUT_DIR / f"cluster_{c}" / cluster_src.name
    if cluster_src.exists():
        shutil.copy2(cluster_src, cluster_dst)
    else:
        print(f"⚠️ Arquivo de heatmap do cluster não encontrado: {cluster_src}")
print(f"Clusters detectados: {clusters}")

# === Copiar arquivos ===
copied, missing = 0, []
for _, row in df.iterrows():
    uid = str(row["userId"])
    cluster = str(row["cluster"])

    img_name = build_image_name(uid, ORDER, EXT)
    src = IMAGES_DIR / img_name
    dst = OUT_DIR / f"cluster_{cluster}" / img_name

    if src.exists():
        shutil.copy2(src, dst)
        copied += 1
    else:
        missing.append(str(src))

print(f"\n✅ Imagens copiadas: {copied}")
if missing:
    print(f"⚠️ Imagens não encontradas: {len(missing)} (mostrando até 10)")
    for m in missing[:10]:
        print("  -", m)

print(f"\nArquivos organizados em: {OUT_DIR.resolve()}")


Usando arquivo de labels: artifacts/user_cluster_labels_umap_order2.csv
Pasta de saída: clusters_2order
Registros carregados: 201
Clusters detectados: [0, 1, 2, 3, 4, 5, 6, 7]

✅ Imagens copiadas: 201

Arquivos organizados em: /Users/otacilio.maia/Desktop/studies/experimentosmestrado/helloworld/clusters_2order
